In [ ]:
from obspy import *
import pandas as pd
import matplotlib.pyplot as plt
from obspy.clients.fdsn import Client
import matplotlib.dates as mdates
import numpy as np
import datetime

In [ ]:
#Small slices: This is a visual check to see if the picker is selecting what want it to
#Very interesting. This is picking both the p and s-wave on the first run here. 

%run vnda_picker_orig.py 2025-11-22T01:44 2025-11-22T01:47:00
#%run palmer_picker.py 2025-11-22T01:44 2025-11-22T01:57:00
#%run vnda_picker_orig.py 2025-11-22T03:47 2025-11-22T03:49:00
#%run vnda_picker_orig.py 2025-11-25T02:08 2025-11-25T02:10:00
#%run vnda_picker_orig.py 2025-11-25T02:08 2025-11-25T02:10:00

In [ ]:
def stat_plotter(df, df1):
    #Calculate hourly/daily metrics
    hourly_counts = df.groupby(df['local'].dt.hour).size() #in local time
    hour_utc = df.groupby(df['onset'].dt.hour).size() #UTC
    daily_counts = df.groupby(df['local'].dt.day).size() #in local time
    daily_utc = df.groupby(df['onset'].dt.day).size()

    #Plot hourly variation of signal detection
    fig, ax = plt.subplots()
    ax.scatter(df['hour_local'].unique(), hourly_counts, label = 'unfiltered') #local time
    ax.set_xlabel('Hours in Local Time (UTC+13)')
    ax.set_ylabel('Number of Triggers')
    #ax.set_ylim(50,200)
    ax.scatter(df1['hour_local'].unique(), df1.groupby(df1['local'].dt.hour).size(), color = 'red', label = 'Filtered for double count') #local time
    ax.legend()
    fig.suptitle('Triggers Grouped by Time of Day unfiltered vs filtered')
    
    #plt.show()
    #this was a check to make sure they look the same and not clipping anything. Looks good. 
    
    #fig, ax = plt.subplots()
    #ax.scatter(df['hour_UTC'].unique(), hour_utc) #UTC time
    #ax.set_xlabel('Hours in UTc Time')
    #ax.set_ylabel('Number of Triggers')
    #ax.set_title('Triggers Grouped by Time of Day (UTC)')
    #plt.show()
    
    
    #pick_5 = daily_utc/(24*60) #event time put y-axis in pick per 5 ish min, or pick per minute
    fig, ax = plt.subplots()
    #ax.scatter(pd.to_datetime(df['onset']).dt.date.unique(), daily_utc)
    ax.scatter(pd.to_datetime(df['onset']).dt.date.unique(), daily_utc/(24), label = 'unfiltered') #pick per hour
    ax.scatter(pd.to_datetime(df['onset']).dt.date.unique(), df1.groupby(df1['onset'].dt.day).size()/24, color = 'red', label = 'Filtered for double count')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
    ax.set_xlabel('Days in UTC Time')
    #ax.set_ylim(5,15)
    ax.set_title("Average Number of Events per Hour")
    ax.set_ylabel('Number of Triggers per Hour')
    ax.legend()
    
    fig.tight_layout(pad=3.0) 

    fig.add_gridspec(4, 4, wspace=0, hspace=0)
   
    #interevent time plot
   
    fig, ax = plt.subplots()
    #ax.scatter(pd.to_datetime(df['onset']).dt.date.unique(), daily_utc)
    ax.hist(df['interevent_time'].dt.total_seconds()/60, bins = 80, edgecolor = 'black', label = 'unfiltered') 
    ax.hist(df1['interevent_time'].dt.total_seconds()/60, bins = 80, edgecolor = 'black', color = 'red', label = 'filtered')
    #ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
    ax.set_xlabel('Time in Minutes')
    ax.set_title("Interevent Spacing")
    ax.set_ylabel('Occurances')
    ax.legend()
    plt.show()

    fig, ax = plt.subplots()

    #ax = df.groupby(pd.Grouper(key='onset', freq='h')).size().plot(color = 'blue')
    hourly_counts = df1.groupby(pd.Grouper(key='onset', freq='h')).size()
    print(type(hourly_counts))
    ax = hourly_counts.plot(color = 'red', label = 'filtered')
    ax.set_xlabel('Days in UTC Time')
    ax.set_title("Number of Events per Hour")
    ax.set_ylabel('Number of Triggers per hour')
    #ax.set_ylim(0, 25)
    fig.tight_layout(pad=3.0) 
    fig.legend()
    fig.add_gridspec(8, 8, wspace=0, hspace=0)

    return None
    

In [ ]:
#Call vnda_picker_orig.py with the associated date range. This will output a list of trigger times. 
#I have the plottnig function of this set to False. Change to true to see the waveform/spectrogram
#CHANGE plot to FALSE otherwise it will be super messy
%run vnda_picker_orig.py 2025-11-21T00:00:00 2025-11-29T00:00:00


In [ ]:
#Trigger dataframe stuff
trigger_df = pd.DataFrame(trig_times)

#trigger_df.set_index()
trigger_df[0] = [t.datetime for t in trigger_df[0]]
trigger_df[1] = [t.datetime for t in trigger_df[1]]
trigger_df['month'] = trigger_df[0].dt.month
trigger_df['day'] = trigger_df[0].dt.day
trigger_df['hour_UTC'] = trigger_df[0].dt.hour

#Times in local Antarctica time, better for daylight tracking, diurnal patterns
trigger_df['local'] = trigger_df[0].dt.tz_localize('UTC').dt.tz_convert('Antarctica/McMurdo')
trigger_df['hour_local'] = trigger_df['local'].dt.hour
trigger_df['day_local'] = trigger_df['local'].dt.day
trigger_df.rename(columns={0:'onset', 1:'offset'}, inplace = True)

#Interevent spacing
trigger_df['interevent_time'] = trigger_df['onset'].shift(-1) - trigger_df['offset']

#Hourly and Daily counts
trigger_df["hourly_counts"] = trigger_df.groupby(trigger_df['local'].dt.hour).size() #in local time
hour_utc = trigger_df.groupby(trigger_df['onset'].dt.hour).size() #UTC
daily_counts = trigger_df.groupby(trigger_df['local'].dt.day).size() #in local time
daily_utc = trigger_df.groupby(trigger_df['onset'].dt.day).size()
trigger_df['amplitude m'] = pd.DataFrame(peak_amps) #amplitude - displacement in meters

#Filtered to no events under 1 minute 
filtered_df = trigger_df[trigger_df['interevent_time'].dt.total_seconds() > 60]

trigger_df

In [ ]:
unfilt_min = trigger_df['interevent_time'].dt.total_seconds()/60
filt_min = filtered_df['interevent_time'].dt.total_seconds()/60
print('unfiltered average interevent spacing:', round(unfilt_min.mean(),2), 'minutes')
print('unfiltered std interevent spacing:', round(unfilt_min.std(),3), 'minutes')
print('unfiltered mode interevent spacing:', round(unfilt_min.mode(),3), 'minutes')

print('filtered average interevent spacing:', round(filt_min.mean(),2), 'minutes')
print('filtered std interevent spacing:', round(filt_min.std(),3), 'minutes')
print('filtered mode interevent spacing:', round(filt_min.mode(),3), 'minutes')


In [ ]:
# COMPARING THE FILTERED VS. UNFILTERED TRIGGER PICKS. FILTERING BY TIME INBETWEEN TRIGGERS
#adding in the additional filter by amplitude reduces the likelihood of double picking!
stat_plotter(trigger_df, filtered_df)


## Preparing for Template Matching ##

In [ ]:
#Dayplots
st.filter("bandpass", freqmin=2, freqmax=10)
for tr in st:
    for i in range(len(pd.to_datetime(trigger_df['onset']).dt.date.unique())):
        start_time = st[0].stats.starttime
        day_start = start_time + (i * 86400)
        day_end = day_start + 86400
        daily_st = st.slice(day_start, day_end)
        local_offset = +13 * 3600
        daily_st.plot(type='dayplot', interval=60, tick_format='%m/%d %Hh', 
                      offset=local_offset,
                      show_y_UTC_label=False  )# interval is minutes per line


In [ ]:
sliced_wvf[slicy]

In [ ]:
#Slices Plotting
from obspy.core.event import (
    Catalog, Event, Pick, WaveformStreamID)
from eqcorrscan.core.match_filter import *
picks = []
for slicy in range(10):
    #print(sliced_wvf[slicy])
    #sliced_wvf[slicy].write(f"{sliced_wvf[slicy].stats.station+ sliced_wvf[slicy].stats.starttime.strftime('%m-%dT%H:%M')}.mseed", format="MSEED")
    #sliced_wvf[slicy].plot()
    #sliced_wvf[slicy].spectrogram()
    picks.append(Pick(sliced_wvf[slicy]))
picks
event = Event(picks = picks)
catalog = Catalog([event])
event
template1 = Template(name = 'wiggle_1', st = picks[0])
tribe = Tribe(templates=[template1])
detections = tribe.detect(stream = st, threshold = 0.5, threshold_type = "MAD", trig_int = 1)

In [ ]:
from eqcorrscan import Tribe

tribe = Tribe().construct(
    method="from_client",
    client_id=bank,
    catalog=cat,
    lowcut=2.0,
    highcut=15.0,
    samp_rate=50.0,
    filt_order=4,
    length=3.0,
    prepick=0.5,
    swin="all",
    process_len=3600,
    all_horiz=True,
    min_snr=4.0,
    parallel=True
)